In [3]:
import csv
import json
import re
import warnings
from pathlib import Path

import spacy

RAW_DIR = Path("../data/raw/daicwoz")
OUT_FILE = Path("../data/processed/daicwoz_finetune.jsonl")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

EXCLUDE_SESSIONS = {451, 458, 480}

SYSTEM_PROMPT = (
    "You are Ellie, a virtual clinical interviewer. "
    "Conduct a structured, empathetic mental health interview by asking open-ended questions "
    "and responds naturally to what the participant shares."
)

NOISE_PATTERN = re.compile(r"<[^>]+>|(?<!\w)xxx(?!\w)", re.IGNORECASE)
TAG_PAREN_PATTERN = re.compile(r"^\w+\s+\((.+)\)$")
SENTENCE_START_PATTERN = re.compile(r"([.!?]\s+)([a-z])")
PRONOUN_I_PATTERN = re.compile(r"\bi\b")
ELLIE_PATTERN = re.compile(r"\bellie\b", re.IGNORECASE)
# Matches underscore-separated single letters: l_a -> LA, p_t_s_d -> PTSD
ABBREV_PATTERN = re.compile(r"\b([a-z])(_[a-z])+\b")

def expand_abbreviations(text: str) -> str:
    return ABBREV_PATTERN.sub(lambda m: m.group(0).replace("_", "").upper(), text)

def clean(text: str) -> str:
    m = TAG_PAREN_PATTERN.match(text.strip())
    if m:
        text = m.group(1)
    text = NOISE_PATTERN.sub("", text)
    text = expand_abbreviations(text)
    return " ".join(text.split()).strip()

def capitalize_sentences(text: str) -> str:
    text = text[:1].upper() + text[1:] if text else text
    return SENTENCE_START_PATTERN.sub(lambda m: m.group(1) + m.group(2).upper(), text)

def fix_pronoun_i(text: str) -> str:
    return PRONOUN_I_PATTERN.sub("I", text)

def fix_ellie(text: str) -> str:
    return ELLIE_PATTERN.sub("Ellie", text)

def capitalize_proper_nouns(text: str, nlp) -> str:
    doc = nlp(text)
    return "".join(
        (token.text if token.text.isupper() else token.text.capitalize()) + token.whitespace_
        if token.pos_ == "PROPN"
        else token.text_with_ws
        for token in doc
    )

def load_conversation(csv_path: Path) -> list[dict]:
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            speaker = row["speaker"].strip()
            value = clean(row["value"])
            if not value:
                continue
            role = "assistant" if speaker == "Ellie" else "user"
            if rows and rows[-1]["role"] == role:
                rows[-1]["content"] += " " + value
            else:
                rows.append({"role": role, "content": value})
    return rows

# --- Load all conversations ---
print("Loading conversations...")
all_conversations = []
for csv_path in sorted(RAW_DIR.glob("*_TRANSCRIPT.csv")):
    session_id = int(csv_path.stem.split("_")[0])
    if session_id in EXCLUDE_SESSIONS:
        print(f"Skipping excluded session {session_id}")
        continue
    turns = load_conversation(csv_path)
    if len(turns) >= 2:
        all_conversations.append(turns)
print(f"Loaded {len(all_conversations)} conversations")

# --- Patch transformers compatibility (idempotent: only patch once per session) ---
from transformers.pipelines import TokenClassificationPipeline

if not getattr(TokenClassificationPipeline, "_grouped_entities_patched", False):
    _orig_sanitize = TokenClassificationPipeline._sanitize_parameters

    def _patched_sanitize(self, **kwargs):
        if "grouped_entities" in kwargs:
            kwargs["aggregation_strategy"] = "simple" if kwargs.pop("grouped_entities") else "none"
        return _orig_sanitize(self, **kwargs)

    TokenClassificationPipeline._sanitize_parameters = _patched_sanitize
    TokenClassificationPipeline._grouped_entities_patched = True

# --- Load models ---
from deepmultilingualpunctuation import PunctuationModel

print("Loading punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    punct_model = PunctuationModel()

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

# --- Collect all turns ---
coords, texts = [], []
for c_idx, turns in enumerate(all_conversations):
    for t_idx, turn in enumerate(turns):
        coords.append((c_idx, t_idx))
        texts.append(turn["content"])

# --- Punctuation restoration + capitalization ---
print(f"Processing {len(texts)} turns...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    restored_texts = [punct_model.restore_punctuation(t) for t in texts]

print("Applying capitalization fixes...")
for (c_idx, t_idx), restored in zip(coords, restored_texts):
    text = capitalize_sentences(restored)
    text = fix_pronoun_i(text)
    text = capitalize_proper_nouns(text, nlp)
    text = fix_ellie(text)
    all_conversations[c_idx][t_idx]["content"] = text
print("Done.")

# --- Write to JSONL ---
records_written = 0
with OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in all_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations to {OUT_FILE}")


Loading conversations...
Skipping excluded session 451
Skipping excluded session 480
Loaded 186 conversations
Loading punctuation model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading spaCy model...
Processing 21039 turns...
Applying capitalization fixes...
Done.
Wrote 186 conversations to ../data/processed/daicwoz_finetune.jsonl
